In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
!uv pip install langchain langchain-openai langchain-community langchain-text-splitters python-dotenv pypdf sentence-transformers faiss-cpu tools deepeval rank-bm25 fitz

Using Python 3.12.13 environment at: /usr
Resolved 142 packages in 1.95s
Prepared 39 packages in 1.23s
Uninstalled 2 packages in 16ms
Installed 39 packages in 161ms
 + acres==0.5.0
 + backoff==2.2.1
 + ci-info==0.4.0
 + configobj==5.0.9
 + configparser==7.2.0
 + dataclasses-json==0.6.7
 + deepeval==3.9.9
 + etelemetry==0.3.1
 + execnet==2.1.2
 + faiss-cpu==1.13.2
 + fitz==0.0.1.dev2
 + langchain-classic==1.0.6
 + langchain-community==0.4.1
 - langchain-core==1.3.1
 + langchain-core==1.3.3
 + langchain-openai==1.2.1
 + langchain-protocol==0.0.15
 + langchain-text-splitters==1.1.2
 + looseversion==1.3.0
 + marshmallow==3.26.2
 + mypy-extensions==1.1.0
 + nipype==1.11.0
 + pathlib==1.0.1
 + portalocker==3.2.0
 + posthog==7.14.0
 + prov==2.1.1
 + puremagic==2.2.0
 + pyfiglet==1.0.4
 + pypdf==6.10.2
 + pytest-asyncio==1.3.0
 + pytest-repeat==0.9.4
 + pytest-rerunfailures==16.1
 + pytest-xdist==3.8.0
 + pyxnat==1.6.4
 + rank-bm25==0.2.2
 + rdflib==7.6.0
 - requests==2.32.4
 + requests==2.33.

In [4]:
import os
import sys
from dotenv import load_dotenv
from langchain_community.docstore.document import Document
from typing import List, Dict, Any, Tuple
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_core.retrievers import BaseRetriever
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from sentence_transformers import CrossEncoder
sys.path.append("/content/drive/MyDrive/Datasets")



load_dotenv("/content/drive/MyDrive/Datasets/API_KEYS.env")
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

from utils.helper_functions import *
from utils.evaluate_rag import *

In [5]:
path = "/content/drive/MyDrive/Datasets/pdfs/Understanding_Climate_Change.pdf"

In [6]:
vectorstore = encode_pdf(path)

# LLM based function to rerank the retrieved documents

In [7]:
class RatingScore(BaseModel):
  relevance_score : float = Field(..., description="The relevance score of document to a query.")

def rerank_documents(query:str, docs:List[Document], top_n:int=3) -> List[Document]:

  prompt_template = PromptTemplate(
      input_variables = ['query', 'doc'],
      template = """On a scale of 1-10, rate the relevance of the following document to the query. Consider the specific context and intent of the query, not just keyword matches.
      Query :{query}
      Document:{doc}
      Relevance Score:"""
  )

  llm = ChatOpenAI(temperature=0, model_name="gpt-4o", max_tokens=4000)
  llm_chain = prompt_template | llm.with_structured_output(RatingScore)

  scored_docs = []
  for doc in docs:
    input_data = {"query":query, "doc":doc.page_content}
    score = llm_chain.invoke(input_data).relevance_score

    try:
      score = float(score)
    except ValueError:
      score = 0
    scored_docs.append((doc, score))

  reranked_docs = sorted(scored_docs, key=lambda x:x[1], reverse=True)
  return [doc for doc, _ in reranked_docs[:top_n]]

## Example usage of the reranking function with a sample query relevant to the document

In [8]:
query = "What are the impacts of climate change on biodiversity?"
initial_docs = vectorstore.similarity_search(query, k=15)
reranked_docs = rerank_documents(query, initial_docs)

print("Top initial documents:")
for i, doc in enumerate(initial_docs[:3]):
  print(f"\nDocument {i+1}:")
  print(doc.page_content[:200]+"...")

print(f"Query:{query}\n")
print("Top reranked documents:")
for i, doc in enumerate(reranked_docs):
  print(f"\nDocument {i+1}:")
  print(doc.page_content[:200]+"...")

Top initial documents:

Document 1:
Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Forests, grasslands, and deserts are 
experiencing shi...

Document 2:
goals. Policies should promote synergies between biodiversity conservation and climate 
action. 
Chapter 10: Climate Change and Human Health 
Health Impacts 
Heat-Related Illnesses 
Rising temperature...

Document 3:
managed retreats. 
Extreme Weather Events 
Climate change is linked to an increase in the frequency and severity of extreme weather 
events, such as hurricanes, heatwaves, droughts, and heavy rainfall...
Query:What are the impacts of climate change on biodiversity?

Top reranked documents:

Document 1:
Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Forests, grasslands, and deserts are 
experiencing shi...

Document 2:
Coral ree

## Create a custom retriever based on our reranker

In [9]:
from pydantic import ConfigDict

class CustomRetriever(BaseRetriever, BaseModel):

  vectorstore : Any = Field(description="vector score for initial retrieval")

  model_config = ConfigDict(arbitrary_types_allowed=True)

  def _get_relevant_documents(self, query:str, num_docs=1) -> List[Document]:
    initial_docs = self.vectorstore.similarity_search(query, k=30)
    return rerank_documents(query, initial_docs, top_n=num_docs)

  async def _aget_relevant_documents(self, query:str) -> List[Document]:
    return self._get_relevant_documents(query)

custom_retriever = CustomRetriever(vectorstore=vectorstore)

llm = ChatOpenAI(temperature=0, model_name="gpt-4o")

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=custom_retriever,
    return_source_documents =True
)

In [10]:
result = qa_chain({"query":query})

print(f"\nQuestion:{query}")
print(f"Answer :{result['result']}")
print("\nRelevant source documents:")
for i, doc in enumerate(result['source_documents']):
  print(f"\nDocument {i+1}:")
  print(doc.page_content[:200]+"...")

/tmp/ipykernel_7210/2527262272.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa_chain({"query":query})



Question:What are the impacts of climate change on biodiversity?
Answer :Climate change impacts biodiversity by shifting habitat ranges, changing species distributions, and affecting ecosystem functions. In terrestrial ecosystems, such as forests, grasslands, and deserts, these changes can lead to a loss of biodiversity and disrupt ecological balance. In marine ecosystems, rising sea temperatures, ocean acidification, and changing currents affect marine biodiversity, leading to species migration and changes in reproductive cycles, which can disrupt marine food webs and fisheries.

Relevant source documents:

Document 1:
Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Forests, grasslands, and deserts are 
experiencing shi...


## Example that demonstrates why we should use reranking

In [11]:
chunks = [
    "The capital of France is great.",
    "The capital of France is huge.",
    "The capital of France is beautiful.",
    """Have you ever visited Paris? It is a beautiful city where you can eat delicious food and see the Eiffel Tower.
    I really enjoyed all the cities in france, but its capital with the Eiffel Tower is my favorite city.""",
    "I really enjoyed my trip to Paris, France. The city is beautiful and the food is delicious. I would love to visit again. Such a great capital city."
]

docs = [Document(page_content=sentence) for sentence in chunks]

def compare_rag_techniques(query:str, docs:List[Document] = docs) -> None:
  from langchain_openai import OpenAIEmbeddings
  from langchain_community.vectorstores import FAISS

  embeddings = OpenAIEmbeddings()
  vectorstore = FAISS.from_documents(docs, embeddings)

  print("Comparision of Retrieval Techniques")
  print("===================================")
  print(f"Query:{query}\n")

  print("Baseline Retrieval Result:")
  baseline_docs = vectorstore.similarity_search(query, k=2)
  for i, doc in enumerate(baseline_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)
  print("Advanced Retrieval Result:")
  custom_retriever = CustomRetriever(vectorstore=vectorstore)
  advanced_docs = custom_retriever.invoke(query)
  for i, doc in enumerate(advanced_docs):
    print(f"\nDocument:{i+1}:")
    print(doc.page_content)

query = "What is the capital of france?"
compare_rag_techniques(query, docs)

Comparision of Retrieval Techniques
Query:What is the capital of france?

Baseline Retrieval Result:

Document 1:
The capital of France is great.

Document 2:
The capital of France is beautiful.
Advanced Retrieval Result:

Document:1:
Have you ever visited Paris? It is a beautiful city where you can eat delicious food and see the Eiffel Tower.
    I really enjoyed all the cities in france, but its capital with the Eiffel Tower is my favorite city.


# Cross Encoder Models

In [12]:
from sentence_transformers import CrossEncoder

model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')
scores = model.predict([
    ("How many people live in Berlin?", "Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers."),
    ("How many people live in Berlin?", "Berlin is well known for its museums."),
])
print(scores)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[ 8.60714   -4.3200803]


In [13]:
from pydantic import ConfigDict

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

class CrossEncoderRetriever(BaseRetriever, BaseModel):
  vectorstore: Any = Field(description = "Vector store for initial retrieval")
  cross_encoder: Any = Field(description = "Cross - encoder model for reranking")
  k: int=Field(default=5, description="Number of documents to retrieve initially")
  rerank_top_k: int=Field(default=3, description="Number of documents to return after reranking")

  model_config = ConfigDict(arbitrary_types_allowed=True)

  def _get_relevant_documents(self, query:str) -> List[Document]:
    #initial retrieval
    initial_docs = self.vectorstore.similarity_search(query, k=self.k)

    #Prepare pairs for cross-encoder
    pairs = [[query, doc.page_content] for doc in initial_docs]

    #Get cross-encoder scores
    scores = self.cross_encoder.predict(pairs)

    #Sort documents by score
    scored_docs = sorted(zip(initial_docs, scores), key=lambda x:x[1], reverse=True)

    #Return top reranked documents
    return [doc for doc, _ in scored_docs[:self.rerank_top_k]]

  async def _aget_relevant_documents(self, query:str) -> List[Document]:
    return self._get_relevant_documents(query)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## Create an instance and showcase over an example

In [14]:
# Create the cross-encoder retriever
cross_encoder_retriever = CrossEncoderRetriever(
    vectorstore=vectorstore,
    cross_encoder=cross_encoder,
    k=10,  # Retrieve 10 documents initially
    rerank_top_k=5  # Return top 5 after reranking
)

# Set up the LLM
llm = ChatOpenAI(temperature=0, model_name="gpt-4o")

# Create the RetrievalQA chain with the cross-encoder retriever
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=cross_encoder_retriever,
    return_source_documents=True
)

# Example query
query = "What are the impacts of climate change on biodiversity?"
result = qa_chain({"query": query})

print(f"\nQuestion: {query}")
print(f"Answer: {result['result']}")
print("\nRelevant source documents:")
for i, doc in enumerate(result["source_documents"]):
    print(f"\nDocument {i+1}:")
    print(doc.page_content[:200] + "...")  # Print first 200 characters of each document


Question: What are the impacts of climate change on biodiversity?
Answer: Climate change impacts biodiversity by altering terrestrial and marine ecosystems. In terrestrial ecosystems, it shifts habitat ranges, changes species distributions, and affects ecosystem functions, leading to potential loss of biodiversity and disruption of ecological balance. In marine ecosystems, rising sea temperatures, ocean acidification, and changing currents affect marine biodiversity, disrupting marine food webs and fisheries. These changes can lead to species migration and altered reproductive cycles, further impacting biodiversity.

Relevant source documents:

Document 1:
Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Forests, grasslands, and deserts are 
experiencing shi...

Document 2:
protection, and habitat creation. 
Climate-Resilient Conservation 
Conservation strategies must account for climate c

# Langchain

In [15]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.chains import RetrievalQA

# Load PDF
loader = PyPDFLoader("/content/drive/MyDrive/Datasets/pdfs/Understanding_Climate_Change.pdf")
documents = loader.load_and_split()

# Create vector store
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(documents, embeddings)

# Startup re-ranker
model = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
compressor = CrossEncoderReranker(model=model, top_n=3)

retriever = vectorstore.as_retriever(search_kwargs={"k":10})
compression_retriever = ContextualCompressionRetriever(
    base_compressor = compressor,
    base_retriever = retriever
)

# Create QA chain
llm = ChatOpenAI(model="gpt-4o-mini")
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=compression_retriever
)

# Query
response = qa_chain.invoke("What are the main causes of climate change?")
print(response)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'query': 'What are the main causes of climate change?', 'result': 'The main causes of climate change include:\n\n1. **Increase in Greenhouse Gases**: The primary cause is the rise in greenhouse gases such as carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O) in the atmosphere. These gases trap heat from the sun, leading to the greenhouse effect.\n\n2. **Burning Fossil Fuels**: Human activities, particularly the burning of fossil fuels (coal, oil, and natural gas) for energy, release large amounts of CO2. This has significantly increased since the industrial revolution.\n\n3. **Deforestation**: Cutting down forests reduces the number of trees that can absorb CO2, exacerbating the greenhouse gas effect.\n\nThese factors combined have intensified the natural processes of climate change, leading to a warmer climate.'}
